# Modell 2 Crossvalidation

### Was kann dieses Modell?

### Daten: Ein Rennen (Belgien)
### Modelle: LogReg, DTree, XGBoost
### Training: Cross-validation
### Vorhersage: Alle Probs für alle Fahrer
### Problem: ???

## Imports

In [1]:
import pandas as pd

In [2]:
import fastf1
from fastf1 import plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# sklearn related imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.utils.class_weight import compute_sample_weight



project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.Feature_Engineer import FeatureEngineer
from utils.Metrics import Metrics
from utils.Models import Models
from utils.experiment_db import init_db
init_db()


pd.set_option('display.width', None)         # passt die Ausgabe-Breite an
pd.set_option('display.max_columns', None)   # zeigt alle Spalten
pd.set_option('display.max_rows', None)      # zeigt alle Reihen

%load_ext autoreload
%autoreload 2

## Rennen definieren

In [3]:
YEAR = 2024
GRAND_PRIX = ["Belgian Grand Prix"]
SESSION_TYPE = "R"

## Daten laden

In [4]:
all_laps = []
for gp in GRAND_PRIX:
    try:
        session = fastf1.get_session(YEAR, gp, SESSION_TYPE)
        session.load(weather=True, laps=True, telemetry=True)
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps = laps.sort_values("Time")
        weather = weather.sort_values("Time")

        merged = pd.merge_asof(
                laps,
                weather,
                on="Time",
                direction="backward"
            )
        
        merged['GP'] = gp
        merged['Year'] = YEAR
        
        all_laps.append(merged)
        print(f"✔ Loaded {gp}")
        
    except Exception as e:
        print(f"Error loading {gp}: {e}")
        
data = pd.concat(all_laps, ignore_index=True)

req         WARNING 	DEFAULT CACHE ENABLED! (2.5 GB) /Users/lukaskoch/Library/Caches/fastf1
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '14'
core        WARNING 	Fixed incorrect tyre stint information for driver '3'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
req            INFO 	Using cached data for car_data
req            INFO 	

✔ Loaded Belgian Grand Prix


## Einheitliche Features laden

In [5]:
fe = FeatureEngineer()
merged = fe.create_features(data)

## Modelle definieren

In [6]:
models = Models.define_models()

## Features auswählen

In [7]:
features = fe.choose_features(merged)
features

['gapAheadSeconds',
 's1_delta',
 's2_delta',
 's3_delta',
 'lap_pace_delta',
 'is_raining',
 'position_change',
 'compound_HARD',
 'compound_MEDIUM',
 'compound_SOFT',
 'status_1',
 'status_12']

In [8]:
X = merged[features]
y = merged['pit_in_next_lap']

## Plot Funktionen

In [9]:
import math
import matplotlib.pyplot as plt
import fastf1.plotting as f1plot

In [10]:
def plot_pit_stop_probabilities(merged, model_name):

    f1plot.setup_mpl(mpl_timedelta_support=True, color_scheme='fastf1')

    # Liste aller Fahrer im Rennen
    drivers = merged['Driver'].unique()
    drivers = [d for d in drivers if isinstance(d, str)]  # falls NaN vorkommen

    # Grid-Einstellungen
    rows, cols = 4, 5
    plots_per_page = rows * cols
    num_pages = math.ceil(len(drivers) / plots_per_page)

    for page in range(num_pages):

        fig, axes = plt.subplots(rows, cols, figsize=(22, 14))
        axes = axes.flatten()

        # Fahrer für diese Seite wählen
        start = page * plots_per_page
        end = start + plots_per_page
        driver_subset = drivers[start:end]

        for ax, driver in zip(axes, driver_subset):

            drv = merged[merged['Driver'] == driver]

            # Stil aus FastF1 holen
            try:
                style = plotting.get_driver_style(
                    identifier=driver, 
                    style=['color', 'linestyle'], 
                    session=session
                )
            except:
                style = {"color": "white", "linestyle": "-"}

            # Probability Plot
            ax.plot(
                drv['LapNumber'],
                drv['pit_stop_prob'],
                label=f'{driver}',
                **style
            )
            #ax.set_yscale('log')

            # Pit Stops markieren
            #pit_laps = drv.loc[drv['Pit_next_lap'].astype(int) == 1, 'LapNumber']
            pit_laps = drv.loc[drv['pit_this_lap'] == 1, 'LapNumber']

            for lap in pit_laps:
                ax.axvline(lap, color='red', linestyle='--', alpha=0.7)

            ax.set_title(f'{driver}')
            ax.set_xlabel('Lap')
            ax.set_ylabel('Pit Stop Prob (log)')

        # Leere Achsen verstecken (falls letzte Seite < 12 Fahrer)
        for ax in axes[len(driver_subset):]:
            ax.axis('off')
            
        fig.suptitle(f"Pit Stop Probability — {model_name}", fontsize=24)
        plt.tight_layout()
        
        save_path = f"plots/plots_model_01/all_{model_name}.png"
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)


In [11]:
def plot_albon_pit_stop_probability(merged, session, model_name):
    fastf1.plotting.setup_mpl(mpl_timedelta_support=True, color_scheme='fastf1')

    albon = merged[merged['Driver'] == 'ALB']
    style = plotting.get_driver_style(identifier='ALB', style=['color', 'linestyle'], session=session)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(albon['LapNumber'], albon['pit_stop_prob'], label='Predicted Pit Stop Probability', **style)
    #ax.set_yscale('log')

    #pit_laps = albon.loc[albon['pit_in_next_three_laps'].astype(int) == 1, 'LapNumber']
    pit_laps = albon.loc[albon['pit_this_lap'] == 1, 'LapNumber']


    for lap in pit_laps:
        ax.axvline(lap, color='red', linestyle='--', alpha=0.7)
        ax.text(lap, ax.get_ylim()[1]*0.8, f'',
                rotation=90, color='red', fontsize=9, ha='center', va='bottom') 

    ax.set_title('Predicted Pit Stop Probability for ALB in 2024 Spa GP')
    ax.set_xlabel('Lap Number')
    ax.set_ylabel('Pit Stop Probability (log scale)')
    max_lap = int(albon['LapNumber'].max())
    ax.set_xticks(range(0, max_lap + 1, 2))
    ax.legend()
    plt.tight_layout()
    
    
    save_path = f"plots/plots_model_01/albon_{model_name}.png"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    #plt.show()

## Modelle laufen lassen

In [12]:
from sklearn.metrics import (
    brier_score_loss, log_loss, roc_auc_score, 
    average_precision_score)

In [13]:
from sklearn.model_selection import StratifiedKFold
from utils.experiment_db import log_or_update_metrics
from utils.Metrics import Metrics

def train_and_evaluate(model, X, y, model_name):
    print(f"\n🔧 Training {model_name} ...")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Sammeln aller Metriken pro Fold
    brier_score_list = []
    logloss_list = []
    rocauc_list = []
    prauc_list = []

    for train_idx, test_idx in skf.split(X, y):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)

        # Vorhersagen
        y_pred_prob = model.predict_proba(X_test)[:, 1]

        # Metriken basierend auf Wahrscheinlichkeiten
        brier_score_list.append(brier_score_loss(y_test, y_pred_prob))
        logloss_list.append(log_loss(y_test, y_pred_prob))
        rocauc_list.append(roc_auc_score(y_test, y_pred_prob))
        prauc_list.append(average_precision_score(y_test, y_pred_prob))


    # Durchschnittswerte
    metrics = {
        "model_name": f"approach_02_{model_name}",
        "brier_score": np.mean(brier_score_list),
        "log_loss": np.mean(logloss_list),
        "roc_auc": np.mean(rocauc_list),
        "pr_auc": np.mean(prauc_list),
    }
    
    # Ausgabe
    print(f"   ✔ Brier Score:              {metrics['brier_score']:.4f} -> optimal 0.0")
    print(f"   ✔ Log-Loss:                 {metrics['log_loss']:.4f} -> optimal 0.0")
    print(f"   ✔ ROC-AUC:                  {metrics['roc_auc']:.4f} -> optimal 1.0")
    print(f"   ✔ PR-AUC:                   {metrics['pr_auc']:.4f} -> optimal 1.0")

    # Logging in die Datenbank
    log_or_update_metrics(
        model_name=f"approach_02_{model_name}",
        metrics={
            'model_name': model_name,
            **metrics
        }
    )

    return metrics


In [14]:
scores = {}
for name, model in models.items():
    metrics = train_and_evaluate(model, X, y, name)
    scores[name] = metrics


🔧 Training LogisticRegression ...
   ✔ Brier Score:              0.2224 -> optimal 0.0
   ✔ Log-Loss:                 0.6282 -> optimal 0.0
   ✔ ROC-AUC:                  0.6042 -> optimal 1.0
   ✔ PR-AUC:                   0.1579 -> optimal 1.0

🔧 Training DecisionTree ...
   ✔ Brier Score:              0.2000 -> optimal 0.0
   ✔ Log-Loss:                 1.4402 -> optimal 0.0
   ✔ ROC-AUC:                  0.4701 -> optimal 1.0
   ✔ PR-AUC:                   0.0473 -> optimal 1.0

🔧 Training xgboost ...
   ✔ Brier Score:              0.0409 -> optimal 0.0
   ✔ Log-Loss:                 0.1759 -> optimal 0.0
   ✔ ROC-AUC:                  0.6235 -> optimal 1.0
   ✔ PR-AUC:                   0.0875 -> optimal 1.0
